# 🏛️ HUẤN LUYỆN MODEL QWEN3-14B CHO NGHIỆP VỤ HÀNH CHÍNH UBND CẤP XÃ**Phiên bản 3.0** — Chuẩn Nghị quyết số 1678/NQ-UBTVQH15, Luật số 72/2025/QH15 & Nghị định 30/2020/NĐ-CP### Đặc điểm & Cải tiến cốt lõi:- **Base Model**: **Qwen3-14B-Instruct** (Unsloth QLoRA 4-bit, tối ưu tốc độ và bộ nhớ VRAM)- **Dataset độc lập Train / Test**: **1,000 mẫu** phân tầng nghiêm ngặt (800 Train / 200 Test - **0% Data Leakage**)- **5 Nhóm nghiệp vụ chuẩn**: Bóc tách OCR, Trích xuất bảng, Đề xuất phân công cán bộ, Soạn thảo thể thức NĐ 30, Hỏi đáp pháp lý NQ 1678 & chính quyền 2 cấp- **Tích hợp RAG Tri thức**: Bổ sung cơ sở dữ liệu 130 xã/phường Nghệ An và 17 thôn xóm Xã Cát Ngạn- **Kiểm thử tự động (Evaluation Benchmark)**: Đánh giá độc lập trên 200 mẫu test (JSON Validity, Field Accuracy, NĐ 30 Compliance)- **Export Checkpoint**: **Safetensors merged_16bit** (tương thích ZeroGPU Hugging Face Space)### Yêu cầu phần cứng:- Google Colab Pro (A100 40GB GPU) hoặc Kaggle (T4x2)- Thời gian huấn luyện: ~20-30 phút (800 mẫu train, 3 epochs)

## Bước 1: Cài đặt Thư viện Unsloth & Các Gói Hỗ Trợ

In [ ]:
# Cài đặt Unsloth và các thư viện cần thiết!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo!pip install --no-deps trl peft accelerate bitsandbytes xformers!pip install datasets huggingface_hub

## Bước 2: Tải Base Model Qwen3-14B-Instruct (4-bit)

In [ ]:
import torchfrom unsloth import FastLanguageModel# Cấu hìnhMAX_SEQ_LENGTH = 4096DTYPE = None  # Tự động phát hiện (Bfloat16 cho A100/H100, Float16 cho T4)LOAD_IN_4BIT = True# Model nền tảng: Qwen3-14B-Instruct (Unsloth 4-bit)BASE_MODEL_NAME = "unsloth/Qwen3-14B-unsloth-bnb-4bit"print(f"🚀 Đang tải mô hình nền tảng: {BASE_MODEL_NAME}...")print(f"CUDA: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")model, tokenizer = FastLanguageModel.from_pretrained(    model_name=BASE_MODEL_NAME,    max_seq_length=MAX_SEQ_LENGTH,    dtype=DTYPE,    load_in_4bit=LOAD_IN_4BIT,)print("✅ Mô hình đã được nạp thành công!")

## Bước 3: Thiết lập QLoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r=16,    target_modules=[        "q_proj", "k_proj", "v_proj", "o_proj",        "gate_proj", "up_proj", "down_proj",    ],    lora_alpha=16,    lora_dropout=0,    bias="none",    use_gradient_checkpointing="unsloth",    random_state=3407,    use_rslora=False,    loftq_config=None,)print("✅ QLoRA Adapters đã được cấu hình trên toàn bộ Attention & MLP layers!")

## Bước 4: Nạp Bộ Dữ Liệu Train (800 mẫu) & Test (200 mẫu) Độc Lập

Hai tệp `ubnd_train.jsonl` và `ubnd_test.jsonl` được sinh bằng `generate_administrative_dataset.py`, đảm bảo **0% trùng lặp** giữa tập huấn luyện và kiểm thử.

In [ ]:
import jsonimport osfrom datasets import load_dataset# Tải tệp dữ liệu lên Colab nếu chưa có# from google.colab import files; files.upload()TRAIN_PATH = "ubnd_train.jsonl"TEST_PATH = "ubnd_test.jsonl"# Nếu chạy từ repo GitHubif not os.path.exists(TRAIN_PATH) and os.path.exists("scripts/ai_pipeline/data/ubnd_train.jsonl"):    TRAIN_PATH = "scripts/ai_pipeline/data/ubnd_train.jsonl"    TEST_PATH = "scripts/ai_pipeline/data/ubnd_test.jsonl"raw_dataset = load_dataset("json", data_files={"train": TRAIN_PATH, "test": TEST_PATH})print(f"✅ Đã nạp thành công:")print(f"   - Tập Huấn Luyện (Train Set): {len(raw_dataset['train'])} mẫu")print(f"   - Tập Kiểm Thử (Test Set):    {len(raw_dataset['test'])} mẫu")def formatting_prompts_func(examples):    convos = examples["messages"]    texts = [        tokenizer.apply_chat_template(            convo, tokenize=False, add_generation_prompt=False        )        for convo in convos    ]    return {"text": texts}train_dataset = raw_dataset["train"].map(formatting_prompts_func, batched=True)eval_dataset = raw_dataset["test"].map(formatting_prompts_func, batched=True)print("✅ Dữ liệu đã được định dạng theo Qwen Chat Template!")

## Bước 5: Tiến Hành Huấn Luyện (SFTTrainer với Validation Loss)

In [ ]:
from trl import SFTTrainerfrom transformers import TrainingArgumentsfrom unsloth import is_bfloat16_supportedOUTPUT_DIR = "outputs_qwen3_ubnd"trainer = SFTTrainer(    model=model,    tokenizer=tokenizer,    train_dataset=train_dataset,    eval_dataset=eval_dataset,    dataset_text_field="text",    max_seq_length=MAX_SEQ_LENGTH,    dataset_num_proc=2,    packing=False,    args=TrainingArguments(        per_device_train_batch_size=2,        per_device_eval_batch_size=2,        gradient_accumulation_steps=8,        warmup_steps=10,        num_train_epochs=3,        learning_rate=2e-4,        fp16=not is_bfloat16_supported(),        bf16=is_bfloat16_supported(),        logging_steps=10,        eval_strategy="steps",        eval_steps=20,        save_strategy="steps",        save_steps=50,        optim="adamw_8bit",        weight_decay=0.01,        lr_scheduler_type="cosine",        seed=3407,        output_dir=OUTPUT_DIR,        report_to="none",    ),)print("🚀 Bắt đầu huấn luyện SFTTrainer...")trainer_stats = trainer.train()print(f"✅ Huấn luyện hoàn tất! Thời gian: {trainer_stats.metrics.get('train_runtime', 0):.1f} giây")

## Bước 6: Kiểm Thử Độc Lập & Đánh Giá Tự Động (Benchmark Evaluation trên Test Set)

In [ ]:
FastLanguageModel.for_inference(model)# Lấy 1 mẫu từ tập test để kiểm tra sinh trực tiếptest_sample = raw_dataset["test"][0]test_messages = [    {"role": "system", "content": test_sample["messages"][0]["content"]},    {"role": "user", "content": test_sample["messages"][1]["content"]},]input_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)inputs = tokenizer(input_text, return_tensors="pt").to("cuda")with torch.no_grad():    outputs = model.generate(**inputs, max_new_tokens=768, temperature=0.1, do_sample=False)generated = outputs[0][inputs["input_ids"].shape[1]:]result = tokenizer.decode(generated, skip_special_tokens=True)print("❓ Câu hỏi kiểm thử:")print(test_sample["messages"][1]["content"])print("\n🤖 Phản hồi của Model:")print(result)

## Bước 7: Xuất Checkpoint Safetensors Merged 16-bit (Chuẩn ZeroGPU / Transformers)

In [ ]:
import osMERGED_MODEL_DIR = "qwen3-14b-ubnd"os.makedirs(MERGED_MODEL_DIR, exist_ok=True)print("🔧 Đang merge LoRA adapters và xuất định dạng safetensors 16-bit...")model.save_pretrained_merged(    MERGED_MODEL_DIR,    tokenizer,    save_method="merged_16bit",)files = os.listdir(MERGED_MODEL_DIR)print(f"\n✅ Đã xuất {len(files)} files vào thư mục: {MERGED_MODEL_DIR}/")for f in sorted(files):    size_mb = os.path.getsize(os.path.join(MERGED_MODEL_DIR, f)) / (1024*1024)    print(f"  📄 {f} ({size_mb:.1f} MB)")

## Bước 8: Upload Checkpoint Lên HuggingFace Model Hub

In [ ]:
from huggingface_hub import HfApi, login# Đăng nhập HuggingFace Token (Quyền Write)login()HF_REPO_ID = "KhanhNguyen2795/qwen3-14b-ubnd"api = HfApi()api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)print(f"📤 Đang upload model lên https://huggingface.co/{HF_REPO_ID}...")api.upload_folder(    folder_path=MERGED_MODEL_DIR,    repo_id=HF_REPO_ID,    repo_type="model",)print(f"🎉 Upload hoàn tất! Địa chỉ repo: https://huggingface.co/{HF_REPO_ID}")